# Project Tycho only has raw cases lets normalize by using population data from the CDC
- https://wonder.cdc.gov/bridged-race-v2020.html

In [4]:
import pandas as pd

population_df = pd.read_csv('../raw/cdc/Bridged-Race Population Estimates 1990-2020.csv')
population_df.head()

,Notes,State,State Code,Yearly July 1st Estimates,Yearly July 1st Estimates Code,Population
0,NaN,Alabama,1.0,1995.0,1995.0,4296800.0
1,NaN,Alabama,1.0,1996.0,1996.0,4331102.0
2,NaN,Alabama,1.0,1997.0,1997.0,4367935.0
3,NaN,Alabama,1.0,1998.0,1998.0,4404701.0
4,NaN,Alabama,1.0,1999.0,1999.0,4430141.0


In [5]:
import us

population_df = population_df[['State', 'Yearly July 1st Estimates', 'Population']]
population_df.dropna(inplace=True)

population_df["State"] = population_df["State"].apply(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)

population_df['Yearly July 1st Estimates'] = population_df['Yearly July 1st Estimates'].astype(int)
population_df.rename(columns={'Yearly July 1st Estimates': 'year', 'State': 'state', 'Population': 'population'}, inplace=True)

population_df.head()

,state,year,population
0,AL,1995,4296800.0
1,AL,1996,4331102.0
2,AL,1997,4367935.0
3,AL,1998,4404701.0
4,AL,1999,4430141.0


In [6]:
population_df.to_csv('../app/data/population.csv', index=False)

In [7]:
assert False

AssertionError: 

# Ok let's test merging tycho cases and population before updating the website

In [8]:
import pandas as pd

population_df = pd.read_csv('../app/data/population.csv')
population_df.head()

,state,year,population
0,AL,1995,4296800.0
1,AL,1996,4331102.0
2,AL,1997,4367935.0
3,AL,1998,4404701.0
4,AL,1999,4430141.0


In [9]:
tycho_df = pd.read_csv('../app/data/tycho_cases.csv')
tycho_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases
0,1995,AK,NaN,13.0,1.0
1,1995,AL,NaN,4.0,38.0
2,1995,AR,2.0,10.0,41.0
3,1995,AZ,10.0,2.0,151.0
4,1995,CA,108.0,206.0,463.0


In [10]:
panel_df = pd.merge(tycho_df, population_df, on=['year', 'state'])
panel_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,population
0,1995,AK,NaN,13.0,1.0,604412.0
1,1995,AL,NaN,4.0,38.0,4296800.0
2,1995,AR,2.0,10.0,41.0,2535399.0
3,1995,AZ,10.0,2.0,151.0,4432499.0
4,1995,CA,108.0,206.0,463.0,31696582.0


In [13]:
diseases = ["measles", "mumps", "pertussis"]

for disease in diseases:
    panel_df[f"{disease}_cases_per_100k"] = (
        panel_df[f"{disease}_cases"] / panel_df["population"] * 100_000
    )

In [16]:
panel_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,population,measles_cases_per_100k,mumps_cases_per_100k,pertussis_cases_per_100k
0,1995,AK,NaN,13.0,1.0,604412.0,NaN,2.150851,0.165450
1,1995,AL,NaN,4.0,38.0,4296800.0,NaN,0.093093,0.884379
2,1995,AR,2.0,10.0,41.0,2535399.0,0.078883,0.394415,1.617102
3,1995,AZ,10.0,2.0,151.0,4432499.0,0.225606,0.045121,3.406656
4,1995,CA,108.0,206.0,463.0,31696582.0,0.340731,0.649912,1.460725
